In [1]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import re
import os
import glob

# display settings
pd.set_option('display.max_columns', None)

In [10]:
folder_path = "../data/1_bronze_raw/munis=city_of_tshwane/city=akasia/type=rent"
file_name = "11007_20260202_171247.parquet"
file_path = os.path.join(folder_path, file_name)

df = pd.read_parquet(file_path)

In [11]:
df

,listing_number,scrape_at,listing_data
0,108782207,2026-02-02T17:12:43.899176,"<div class=""p24_tileContainer js_resultTile Se..."
1,116872499,2026-02-02T17:12:43.899994,"<div class=""SeoulForecast js_resultTile p24_ti..."
2,116702498,2026-02-02T17:12:43.900691,"<div class=""SeoulLiberal js_resultTile p24_til..."
3,116850269,2026-02-02T17:12:43.901314,"<div class=""SeoulSaddest js_resultTile p24_til..."
4,116794164,2026-02-02T17:12:43.901769,"<div class=""SeoulYankee js_resultTile p24_tile..."
5,115999979,2026-02-02T17:12:43.902314,"<div class=""SeoulRobe js_resultTile p24_tileCo..."
6,116833744,2026-02-02T17:12:43.902922,"<div class=""SeoulGetting js_resultTile p24_til..."
7,116850243,2026-02-02T17:12:43.903554,"<div class=""SeoulPalette js_resultTile p24_til..."
8,116735719,2026-02-02T17:12:43.904037,"<div class=""SeoulConcord js_resultTile p24_til..."


In [12]:
lookup = pd.read_parquet("../data/0_metadata/suburb_lookup.parquet")
lookup

,suburb_id,suburb_name,city_id,city_name,province_id,province_name,municipality,last_scraped
0,3990,amandasig,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
1,4002,chantelle,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
2,4012,clarina,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
3,4102,doreg ah,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
4,429,eldorette,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
...,...,...,...,...,...,...,...,...
2068,346,rooiwal ah,2473,wonderboom,1,gauteng,City of Tshwane,2026-02-06
2069,347,vasfontein ah,2473,wonderboom,1,gauteng,City of Tshwane,2026-02-06
2070,17,walmansthal ah,2473,wonderboom,1,gauteng,City of Tshwane,2026-02-06
2071,55,waterval ah,2473,wonderboom,1,gauteng,City of Tshwane,None


In [13]:
suburb_id = file_name.split('_')[0]
suburb, city, munis = lookup.loc[lookup['suburb_id'] == suburb_id, ['suburb_name', 'city_name', 'municipality']].values[0]

In [14]:
print(suburb, city, munis)

heatherview akasia City of Tshwane


In [7]:
BeautifulSoup(df["listing_data"][0], "html.parser")

<div class="CrammerLaybys js_resultTile p24_tileContainer" data-listing-number="116872112">
<div class="p24_regularTile js_rollover_container js_resultTileClickable p24_tileHoverWrapper" data-listing-number="116872112" itemscope="" itemtype="http://schema.org/Product">
<meta content="2 Bedroom Apartment" itemprop="name"/>
<div class="" title="2 Bedroom Apartment / flat to rent in Buffelsdrift AH - Pretoria">
<span class="js_listingTileImageHolder p24_image">
<a href="/to-rent/buffelsdrift-ah/pretoria/gauteng/1/116872112">
<img alt="This unit is on the 1st Floor of Oscars Place which is situated in the Paramount Estate Tygerpoort Pretoria East.  Easy access to ..." class="js_rollover_target js_rollover_default js_P24_listingImage js_lazyLoadImage" height="212" itemprop="image" src="https://images.prop24.com/372997007/Crop600x400" title="2 Bedroom Apartment / flat to rent in Buffelsdrift AH - Pretoria" width="318"/>
</a>
</span>
<span class="p24_branding js_tilePseudoLink" data-href="/es

In [ ]:
def clean_numeric(text, size = 1):
    """Helper to turn 'R 8 500' into 8500 or '80 m²' into 80."""
    size_multiplier = 1.0
    if not text: return None
    if 'ha' in text:
        num = text[:-3]
        try:
            num = float(num)
            return num * 10000
        except ValueError as e:
            print(f"error: {e}")
    
    if 'per' in text:
        size_multiplier = size if size else 1

    nums = re.sub(r'[^\d\.]', '', text)
    return float(nums) * size_multiplier if nums else None

def get_flags(description):
    desc = description.lower()
    return {
        "is_sharing" : 1 if any(x in desc for x in ['sharing', 'share']) else 0,
        "is_estate": 1 if "estate" in desc else 0,
        "has_garden": 1 if "garden" in desc else 0,
        "pet_friendly": 1 if "pet" in desc else 0,
        "is_modern": 1 if any(x in desc for x in ["modern", "newly", "renovated"]) else 0
    }

def get_property_type(text):
        """
        Feature engineering from title to get property type.
        """
        if text in ['Commercial Property', 'Industrial Property']:
            return text
        if 'Apartment' in text:
            return 'Apartment'
        if 'Townhouse' in text:
            return 'Townhouse'
        if 'House' in text:
            return 'House'
        return text

def extract_features(df_listings, suburb, city, munis):
    results = []
    for listing in df_listings.itertuples():
        soup = BeautifulSoup(listing.listing_data, "html.parser")

        # Property ID (from the link)
        link_tag = soup.find('a', href=True)
        prop_id = link_tag['href'].split('/')[-1].split('?')[0] if link_tag else None

        title_tag = soup.select_one(".p24_title")
        title = title_tag.text.strip() if title_tag else None

        prop_type = get_property_type(title) if title else None
        
        # 1. Basic Cleaning
        floor_size = clean_numeric(soup.select_one(".p24_size").text if soup.select_one(".p24_size") else None)
        price = clean_numeric(soup.select_one(".p24_price").text if soup.select_one(".p24_price") else None, floor_size)
        
        # 2. Icon Extraction
        features = {f.get('title', '').lower(): f.find('span').text.strip() 
                    for f in soup.find_all('span', class_='p24_featureDetails') if f.find('span')}

        # 3. Description & Flags
        description = soup.select_one(".p24_excerpt").text.strip() if soup.select_one(".p24_excerpt") else ""
        flags = get_flags(description)

        # 4. Assembly
        results.append({
            "property_id": prop_id,
            "price": price,
            "title": title,
            "type": prop_type,
            "suburb": suburb,
            "city": city,
            "municipality": munis,
            "bedrooms": clean_numeric(features.get("bedrooms")),
            "bathrooms": clean_numeric(features.get("bathrooms")),
            "parking": clean_numeric(features.get("parking spaces")),
            "floor_size": floor_size,
            **flags # Merges the dictionary of flags here
        })
    return results

test = extract_features(df, suburb, city, munis)

In [20]:
test

[{'property_id': '108782207',
  'price': 13200.0,
  'title': None,
  'suburb': 'heatherview',
  'city': 'akasia',
  'municipality': 'City of Tshwane',
  'bedrooms': 3.0,
  'bathrooms': 2.0,
  'parking': 2.0,
  'floor_size': 135.0,
  'is_sharing': 0,
  'is_estate': 0,
  'has_garden': 0,
  'pet_friendly': 0,
  'is_modern': 0},
 {'property_id': '116872499',
  'price': None,
  'title': '3 Bedroom House',
  'suburb': 'heatherview',
  'city': 'akasia',
  'municipality': 'City of Tshwane',
  'bedrooms': 3.0,
  'bathrooms': 2.0,
  'parking': 2.0,
  'floor_size': 2635.0,
  'is_sharing': 0,
  'is_estate': 0,
  'has_garden': 0,
  'pet_friendly': 0,
  'is_modern': 0},
 {'property_id': '116702498',
  'price': 7000.0,
  'title': '2 Bedroom House',
  'suburb': 'heatherview',
  'city': 'akasia',
  'municipality': 'City of Tshwane',
  'bedrooms': 2.0,
  'bathrooms': 1.0,
  'parking': 2.0,
  'floor_size': 300.0,
  'is_sharing': 0,
  'is_estate': 1,
  'has_garden': 0,
  'pet_friendly': 0,
  'is_modern': 

In [42]:
# print(clean_numeric("\n\r\n                    R 10\r\n                        per m²\n", 23994))
print(clean_numeric("2.5"))

2.5


In [44]:
lookup['municipality'].unique()

array(['City of Tshwane', 'Ekurhuleni', 'City of Johannesburg',
       'West Rand', 'Sedibeng'], dtype=object)

In [54]:
files = glob.glob("../data/1_bronze_raw/munis=sedibeng/city=*/type=rent/*.parquet")

In [56]:
files

['../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/3589_20260202_234225.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/17142_20260202_234308.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/3675_20260202_234334.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/16725_20260202_234424.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/3815_20260202_234454.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/17112_20260202_234524.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/3816_20260202_234554.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/17412_20260202_234653.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/3721_20260202_234719.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/3598_20260202_234746.parquet',
 '../data/1_bronze_raw/munis=sedibeng/city=evaton/type=rent/12777_20260202_234830.parq

In [34]:
folder_path = "../data/1_bronze_raw/munis=sedibeng/city=vereeniging/type=rent"
file_name = "3701_20260206_085111.parquet"
file_path = os.path.join(folder_path, file_name)

df = pd.read_parquet(file_path)
suburb_id = file_name.split('_')[0]
suburb, city, munis = lookup.loc[lookup['suburb_id'] == suburb_id, ['suburb_name', 'city_name', 'municipality']].values[0]

test_2 = extract_features(df, suburb, city, munis)

In [24]:
text = df.loc[df['listing_number']=='116858222', 'listing_data'].values[0]
soup_t = BeautifulSoup(text)
soup_t.select_one(".p24_price").text

'\n\r\n                    R 10\r\n                        per m²\n'

In [35]:
test_2

[{'property_id': '116859704',
  'price': 10900,
  'suburb': 'duncanville',
  'city': 'vereeniging',
  'municipality': 'Sedibeng',
  'bedrooms': 3,
  'bathrooms': 2,
  'parking': 2,
  'floor_size': None,
  'is_sharing': 0,
  'is_estate': 0,
  'has_garden': 0,
  'pet_friendly': 0,
  'is_modern': 0},
 {'property_id': '116298038',
  'price': 1308090,
  'suburb': 'duncanville',
  'city': 'vereeniging',
  'municipality': 'Sedibeng',
  'bedrooms': None,
  'bathrooms': None,
  'parking': None,
  'floor_size': 37374,
  'is_sharing': 0,
  'is_estate': 0,
  'has_garden': 0,
  'pet_friendly': 0,
  'is_modern': 0},
 {'property_id': '116778508',
  'price': 1225000,
  'suburb': 'duncanville',
  'city': 'vereeniging',
  'municipality': 'Sedibeng',
  'bedrooms': None,
  'bathrooms': None,
  'parking': None,
  'floor_size': 35000,
  'is_sharing': 0,
  'is_estate': 0,
  'has_garden': 0,
  'pet_friendly': 0,
  'is_modern': 0},
 {'property_id': '116872606',
  'price': 13000,
  'suburb': 'duncanville',
  'c